# Additional Experiments - Cross-Stadium Generalization

Evaluation notebook for the "additional experiments" chapter: the pre-quarter-fold
generalization attempts (external stadium data -> fixed TB2026 test set / fixed split).

Each section follows **Method / Results / Discussion**:
- *Method*: what was trained, on what data, evaluated how.
- *Results*: table + chart pulled directly from the experiment folders' `metrics.json` / `kpt_eval.json`.
- *Discussion*: prompts to fill in - some pre-filled with the key numeric finding.

Lives in `Field_Keypoint_Detection/evaluation/` — run with its working directory
set there (default when opened directly in Jupyter/VS Code); reads results from
`Field_Keypoint_Detection/*_results/` one level up.


## Setup

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'savefig.facecolor': 'white',
})

BASE = Path('..')               # Field_Keypoint_Detection/ — where *_results/ live
OUT  = Path('.') / "evaluation_output"  # evaluation/evaluation_output/
OUT.mkdir(exist_ok=True)

# Same flat-UI palette as Player_Detection/evaluation.ipynb and main_evaluation.ipynb,
# assigned per entity below and reused across sections for the same underlying model
# (e.g. "Stage 2" is always blue, whether in the ladder or the per-quarter breakdown).
PALETTE = {
    "blue":   "#4C9BE8",
    "red":    "#e74c3c",
    "green":  "#2ecc71",
    "orange": "#f39c12",
    "purple": "#9b59b6",
    "grey":   "#95a5a6",
}


def load_json(path):
    """Returns parsed JSON or None if the file doesn't exist (incomplete/pending run)."""
    p = BASE / path
    return json.loads(p.read_text()) if p.exists() else None


def kpt_overall(path):
    """overall block of a kpt_eval.json (det_rate, mean/median/p90 pixel error)."""
    d = load_json(path)
    return d["overall"] if d else None


def hash_group_stats(path):
    """Aggregate stats restricted to the 18 hash-mark keypoints (kpt_id 45-62)."""
    d = load_json(path)
    if d is None:
        return None
    n_gt = n_det = 0
    errs = []
    for kpt in d["per_keypoint"].values():
        if "Hash" in kpt.get("group", ""):
            n_gt += kpt["n_gt"]
            n_det += kpt["n_detected"]
            if kpt["n_detected"] > 0:
                errs.append(kpt["mean_err_px"])
    return {
        "det_rate": n_det / n_gt if n_gt else None,
        "mean_err_px": sum(errs) / len(errs) if errs else None,
        "n_gt": n_gt,
    }


def bar(ax, labels, values, colors, ylabel, title):
    bars = ax.bar(labels, values, color=colors, width=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    for b, v in zip(bars, values):
        if v is not None:
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3g}",
                     ha="center", va="bottom", fontsize=9)
    return bars


## 1. Progressive Fine-Tuning Ladder

**Method.** `cross_stadium_experiment.py` — evaluated on the same fixed TB2026 test
set (2920 frames):
- Stage 2: trained on condition1 (game1+2+5).
- Stage 3: Stage 2 -> fine-tuned on ravelin2025.

Stage 1 (raw COCO `yolo11l-pose` weights, zero-shot) is **dropped**: the COCO pose
head outputs 17 human keypoints, while this task has 62 field-landmark keypoints —
Ultralytics' validator can't reshape one into the other (`RuntimeError: view size is
not compatible...`), and conceptually a generic human-pose model has no notion of
"hash mark #47" to begin with. Stage 2 (condition1-only, zero TB2026 data) is the
effective floor of this ladder instead.


In [2]:
STAGE_LABELS = {
    "2": "Stage 2\n+condition1",
    "3": "Stage 3\n+ravelin2025",
}
STAGE_COLORS = {"2": PALETTE["blue"], "3": PALETTE["red"]}

rows = []
for stage, label in STAGE_LABELS.items():
    overall = kpt_overall(f"cross_stadium_results/{stage}/train/kpt_eval.json")
    metrics = load_json(f"cross_stadium_results/{stage}/metrics.json")
    rows.append({
        "stage": label.replace("\n", " "),
        "det_rate": overall["det_rate"] if overall else None,
        "mean_err_px": overall["mean_err_px"] if overall else None,
        "med_err_px": overall["med_err_px"] if overall else None,
        "box_mAP50": metrics["box"]["mAP50"] if metrics else None,
    })
df_ladder = pd.DataFrame(rows)
df_ladder


,stage,det_rate,mean_err_px,med_err_px,box_mAP50
0,Stage 2 +condition1,0.6077,203.78,100.63,0.0990
1,Stage 3 +ravelin2025,0.5873,341.25,166.40,0.0681


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = [l.replace("\n", " ") for l in STAGE_LABELS.values()]
colors = list(STAGE_COLORS.values())
bar(axes[0], labels, df_ladder["det_rate"], colors, "Detection rate", "Keypoint detection rate")
bar(axes[1], labels, df_ladder["mean_err_px"], colors, "Mean error (px)", "Mean localization error")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=0, labelsize=8)
fig.tight_layout()
plt.savefig(OUT / "ladder_stages.png", dpi=150, bbox_inches="tight")
plt.show()


/tmp/ipykernel_3092/3519129940.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion.**
- Stage 2 (condition1-only) is the effective floor: 0.608 det_rate / 204px mean error
  with zero TB2026 data.
- Stage 2 -> Stage 3 det_rate **drops** and mean error **roughly doubles** — fine-tuning
  on ravelin2025 hurts rather than helps. Candidate explanation: catastrophic forgetting
  of condition1 (see Section 2 for the joint-training counter-evidence).
- [ ] Add your own read on *why* ravelin2025 fine-tuning specifically hurts (domain gap
  between ravelin2025 and TB2026? too few ravelin2025 epochs/frames? stadium-specific
  field markings?).


## 2. Joint vs. Sequential Training

**Method.** `combined_training.py` / `combined_training_perspective.py` — train on
condition1 + ravelin2025 *simultaneously* instead of the Stage 2 -> Stage 3 sequential
fine-tune, to test whether joint training avoids the forgetting seen above. The
perspective variant adds `perspective=0.001, degrees=5` to simulate camera-angle
variation across stadiums. Both evaluated on the same TB2026 test set.


In [4]:
JOINT_CONFIGS = [
    ("Stage 3\n(sequential)", "cross_stadium_results/3/train/kpt_eval.json", PALETTE["red"]),
    ("Combined\n(joint)", "combined_results/train/kpt_eval.json", PALETTE["green"]),
    ("Combined\n+perspective aug", "combined_perspective_results/train/kpt_eval.json", PALETTE["orange"]),
]

rows = []
for label, path, color in JOINT_CONFIGS:
    o = kpt_overall(path)
    rows.append({"config": label.replace("\n", " "), **o})
df_joint = pd.DataFrame(rows)
df_joint


,config,n_frames,n_gt_total,n_det_total,det_rate,mean_err_px,med_err_px,p90_err_px,mean_err_norm
0,Stage 3 (sequential),2920,65286,38342,0.5873,341.25,166.40,966.04,0.15491
1,Combined (joint),2920,65286,56003,0.8578,260.76,139.46,726.57,0.11837
2,Combined +perspective aug,2920,65286,48322,0.7402,242.68,168.25,556.08,0.11016


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = [c[0].replace("\n", " ") for c in JOINT_CONFIGS]
colors = [c[2] for c in JOINT_CONFIGS]
bar(axes[0], labels, df_joint["det_rate"], colors, "Detection rate", "Keypoint detection rate")
bar(axes[1], labels, df_joint["mean_err_px"], colors, "Mean error (px)", "Mean localization error")
for ax in axes:
    ax.tick_params(axis="x", labelsize=8)
fig.tight_layout()
plt.savefig(OUT / "joint_vs_sequential.png", dpi=150, bbox_inches="tight")
plt.show()


/tmp/ipykernel_3092/1114001414.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion.**
- Joint training clearly beats sequential fine-tuning (det_rate 0.858 vs 0.587,
  mean error 261px vs 341px) — consistent with the catastrophic-forgetting story from
  Section 1.
- Perspective augmentation *hurts* joint training (det_rate drops to 0.740). [ ] Discuss
  why — for a keypoint task the exact field geometry matters, and `perspective`/`degrees`
  augmentation may distort the geometric relationships the model needs to learn, unlike
  in generic object detection.


## 3. Learning Curve

**Method.** Two distinct curves — keep them clearly separated in the write-up:
- `condition1_learning_curve.py`: scales the amount of *external* (condition1)
  training data; val is the fixed condition1 held-out clips; reported metric is on
  the TB2026 test set.
- `learning_curve.py`: scales the amount of *TB2026's own* data, using a **fixed
  random 20% val split** (not quarter-based — a leakage risk this section should flag).


In [6]:
FRACS = ["frac_005", "frac_010", "frac_025", "frac_050", "frac_075", "frac_100"]

rows = []
for frac in FRACS:
    m = load_json(f"condition1_lc_results/{frac}/metrics.json")
    if m:
        rows.append({
            "frac": frac,
            "n_train_frames": m.get("n_train_frames"),
            "box_mAP50": m["box"]["mAP50"],
            "precision": m["box"]["precision"],
            "recall": m["box"]["recall"],
        })
df_lc_external = pd.DataFrame(rows)
df_lc_external


,frac,n_train_frames,box_mAP50,precision,recall
0,frac_005,38,0.0246,0.0514,0.1777
1,frac_010,129,0.0169,0.0535,0.1990
2,frac_025,204,0.0262,0.0940,0.1353
3,frac_050,455,0.1019,0.2352,0.1976
4,frac_075,680,0.1176,0.2957,0.1925
5,frac_100,870,0.0990,0.2845,0.2052


In [7]:
rows = []
for frac in FRACS:
    m = load_json(f"learning_curve_results/{frac}/metrics.json")
    o = kpt_overall(f"learning_curve_results/{frac}/train/kpt_eval.json")
    if m and o:
        rows.append({
            "frac": frac,
            "n_train_frames": m.get("n_train_frames"),
            "box_mAP50": m["box"]["mAP50"],
            "det_rate": o["det_rate"],
            "mean_err_px": o["mean_err_px"],
            "med_err_px": o["med_err_px"],
        })
df_lc_internal = pd.DataFrame(rows)
df_lc_internal


,frac,n_train_frames,box_mAP50,det_rate,mean_err_px,med_err_px
0,frac_005,116,0.0821,0.2940,467.14,375.94
1,frac_010,244,0.2666,0.8209,405.26,311.70
2,frac_025,612,0.1302,0.8170,357.73,253.00
3,frac_050,1136,0.1026,0.7750,263.59,141.37
4,frac_075,1708,0.0526,0.9285,58.84,14.11
5,frac_100,2332,0.1109,0.8550,33.00,9.49


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5))

axes[0].plot(df_lc_external["n_train_frames"], df_lc_external["box_mAP50"],
             marker="o", color=PALETTE["purple"], linewidth=2, markersize=7,
             label="condition1 (external)")
axes[0].set_xlabel("n training frames")
axes[0].set_ylabel("box mAP50")
axes[0].set_title("External-data learning curve")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(df_lc_internal["n_train_frames"], df_lc_internal["det_rate"],
             marker="o", color=PALETTE["blue"], linewidth=2, markersize=7,
             label="TB2026 (fixed split)")
axes[1].set_xlabel("n training frames")
axes[1].set_ylabel("keypoint detection rate")
axes[1].set_title("In-domain (TB2026) learning curve")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.savefig(OUT / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


/tmp/ipykernel_3092/794659723.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion.**
- External-data curve: box mAP50 stays low and noisy (0.017-0.12) with no clean upward
  trend as condition1 data grows — more external data does not reliably transfer to
  TB2026 performance.
- In-domain curve: det_rate climbs fairly consistently from 0.29 (5%) to 0.86 (100%) —
  TB2026-specific data matters far more than external data volume.
- [ ] Flag the fixed 20%-val-split leakage risk here (frames from the same play/clip
  can land in both train and val) as part of the motivation for the quarter-based folds
  used in the main evaluation (Ch. 6).


## 4. Ablation Study

**Method.**
- `flip_ablation.py`: `no_flip` trains a fresh condition1 model with `fliplr=0.0`;
  `with_flip` reuses the existing Stage 2 weights (trained with the default
  `fliplr=0.5`) — note this means **`with_flip` has no separate weights folder**, its
  `kpt_eval` numbers are identical to `cross_stadium_results/2`.
- `hash_only_training.py`: trains on condition1+ravelin2025 but supervises only the 18
  hash-mark keypoints (all others get `visibility=0`). Compared against the fully
  supervised `combined_results` model, restricted to just the hash-mark subset, to see
  whether narrowing supervision helps the keypoints it targets.


In [9]:
FLIP_CONFIGS = [
    ("no_flip", "flip_ablation_results/no_flip/metrics.json", PALETTE["grey"]),
    ("with_flip (=stage2)", "flip_ablation_results/with_flip/metrics.json", PALETTE["blue"]),
]
rows = []
for label, path, color in FLIP_CONFIGS:
    m = load_json(path)
    rows.append({"config": label, "box_mAP50": m["box"]["mAP50"],
                 "precision": m["box"]["precision"], "recall": m["box"]["recall"]})
df_flip = pd.DataFrame(rows)
df_flip


,config,box_mAP50,precision,recall
0,no_flip,0.0843,0.2123,0.1740
1,with_flip (=stage2),0.0990,0.2845,0.2052


In [10]:
rows = []
for label, path in [("hash_only (18 kpts supervised)", "hash_only_results/train/kpt_eval.json"),
                     ("combined (full supervision)", "combined_results/train/kpt_eval.json")]:
    overall = kpt_overall(path)
    hash_only_stats = hash_group_stats(path)
    rows.append({
        "model": label,
        "overall_det_rate": overall["det_rate"],
        "overall_mean_err_px": overall["mean_err_px"],
        "hash_group_det_rate": hash_only_stats["det_rate"],
        "hash_group_mean_err_px": hash_only_stats["mean_err_px"],
    })
df_hash = pd.DataFrame(rows)
df_hash


,model,overall_det_rate,overall_mean_err_px,hash_group_det_rate,hash_group_mean_err_px
0,hash_only (18 kpts supervised),0.2063,377.11,0.642622,455.331111
1,combined (full supervision),0.8578,260.76,0.898907,227.915000


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

bar(axes[0], df_flip["config"], df_flip["box_mAP50"],
    [c[2] for c in FLIP_CONFIGS], "box mAP50", "Flip augmentation ablation")
axes[0].tick_params(axis="x", labelsize=8)

labels = ["hash_only", "combined\n(full)"]
bar(axes[1], labels, df_hash["hash_group_det_rate"],
    [PALETTE["red"], PALETTE["green"]], "Detection rate (hash kpts only)",
    "Hash-mark-only accuracy,\nrestricted-supervision vs full model")
axes[1].tick_params(axis="x", labelsize=8)

fig.tight_layout()
plt.savefig(OUT / "ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()


/tmp/ipykernel_3092/3842376914.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion.**
- Flip augmentation helps: `with_flip` beats `no_flip` on box mAP50 (0.099 vs 0.084)
  and recall (0.205 vs 0.174).
- Hash-only supervision *does not even help the keypoints it targets*: restricted to
  just the 18 hash-mark keypoints, `hash_only` still trails the fully-supervised
  `combined_results` model (det_rate 0.643 vs 0.899, mean error 455px vs 228px). [ ]
  Discuss why — likely the model relies on the easier/well-supervised keypoints
  (sidelines, numbers) as spatial context to localize the harder hash marks, so removing
  their supervision removes useful signal rather than freeing up capacity.


## 5. Per-Quarter Breakdown

**Method.** `cross_stadium_quarter_eval.py` — no training, re-evaluates the existing
Stage 2 / Stage 3 weights separately on each TB2026 quarter (Q1-Q4), using the same
quarter boundaries as the quarter-based k-fold in Ch. 6.


In [12]:
QUARTERS = ["Q1", "Q2", "Q3", "Q4"]
STAGE_Q_COLORS = {"stage2": PALETTE["blue"], "stage3": PALETTE["red"]}

rows = []
for stage in ["stage2", "stage3"]:
    for q in QUARTERS:
        o = kpt_overall(f"cross_stadium_quarter_results/{stage}/{q}/kpt_eval.json")
        rows.append({"stage": stage, "quarter": q, **o})
df_quarter = pd.DataFrame(rows)
df_quarter


,stage,quarter,n_frames,n_gt_total,n_det_total,det_rate,mean_err_px,med_err_px,p90_err_px,mean_err_norm
0,stage2,Q1,744,16572,7380,0.4453,170.43,91.65,423.13,0.07736
1,stage2,Q2,652,16485,11740,0.7122,191.23,89.70,569.05,0.08681
2,stage2,Q3,748,15827,10128,0.6399,222.35,112.10,580.01,0.10094
3,stage2,Q4,776,16402,10429,0.6358,223.59,108.95,609.93,0.10150
4,stage3,Q1,744,16572,10149,0.6124,408.68,262.58,974.52,0.18552
5,stage3,Q2,652,16485,9421,0.5715,358.59,136.61,1025.61,0.16278
6,stage3,Q3,748,15827,8822,0.5574,303.16,136.93,858.25,0.13762
7,stage3,Q4,776,16402,9951,0.6067,289.94,146.94,793.87,0.13162


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5))
width = 0.35
x = range(len(QUARTERS))

for i, stage in enumerate(["stage2", "stage3"]):
    sub = df_quarter[df_quarter["stage"] == stage].set_index("quarter").loc[QUARTERS]
    offset = (i - 0.5) * width
    axes[0].bar([xi + offset for xi in x], sub["det_rate"], width=width,
                color=STAGE_Q_COLORS[stage], label=stage)
    axes[1].bar([xi + offset for xi in x], sub["mean_err_px"], width=width,
                color=STAGE_Q_COLORS[stage], label=stage)

for ax, title, ylabel in [(axes[0], "Detection rate by quarter", "Detection rate"),
                           (axes[1], "Mean error by quarter", "Mean error (px)")]:
    ax.set_xticks(list(x))
    ax.set_xticklabels(QUARTERS)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.savefig(OUT / "per_quarter_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()


/tmp/ipykernel_3092/2857646856.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion.**
- Stage2 det_rate ranges 0.445 (Q1) to 0.712 (Q2) — a ~27pp spread the single
  aggregated TB2026 number (0.608) completely hides. Stage3 is more stable (0.557-0.612)
  but uniformly worse than Stage2's best quarters.
- [ ] Speculate on *why* Q1 is the hardest quarter for Stage2 (lighting? camera angle
  change mid-game? fewer clips?) if you have that metadata.
- **Bridge to Ch. 6**: this per-quarter variance is exactly why the main evaluation
  moves from one fixed TB2026 test set to quarter-based 4-fold cross-validation — a
  single aggregate number was hiding as much as ~30 percentage points of quarter-to-quarter
  variation.


## Synthesis

- [ ] One paragraph: cross-stadium transfer alone plateaus below what's needed
  (Stage 3 det_rate 0.587 vs. the quarter-based main models' ~0.85-0.93, see Ch. 6),
  joint training partially recovers this but still lags in-domain training, and
  per-quarter variance plus fixed-split leakage risk motivated moving to quarter-based
  CV for the main evaluation.
